## Python Interview Questions

### The goal of this exercise is to explore and clean a typical client dataset with Python

In [1]:
filename = 'python_exploration_data.xlsx'

#### 1. Load the data and examine its contents

In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel(filename)
print(df.shape)
df.head()

(942, 20)


,Region,Neighborhood Leadership Team,Date Submitted:,First,Last,Email Address,Phone,Address,City,State,Zip,School 1,School 2,School 3,Status,Membership Level,Events,Notes,Unnamed: 18,Unnamed: 19
0,NaN,NaN,NaT,Aaron,Zuaiter,8835329@gmail.com,NaN,427,Manhattan Beach,CA,90266,KIPP LA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaT,Adele,Zinn,a.chivagirl@gmail.com,(510) 295-9094,3026,Los Angeles,CA,90045,NaN,NaN,NaN,Parent,1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaT,Adn,Zindel,acapellagirl@hotmail.com,(310) 770-5447,7863,Los Angeles,CA,90049,Kenter,NaN,NaN,Parent,1,NaN,NaN,NaN,NaN
3,NaN,NaN,NaT,Adrian,Zavala,acbrown42@outlook.com,NaN,1 Water Mill,Los angeles,CA,90045,WISH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaT,Adrienne,Zaret,acclee@yahoo.com,(310) 555-5555,10 Serene Lake Place,Los Angeles,CA,90045,Open Magnet Charter School,El Segundo Middle School,El Segundo High School,NaN,NaN,NaN,added 4/28 JK,NaN,NaN


#### 2. How many records do we have per state? How many per school?

### State

In [3]:
df.groupby(['State']).size()

State
90024             1
90049             1
90066             1
90077             1
90272             2
CA              877
CA 90292 USA      1
Ca 90272 USA      2
NY                1
Ohio              1
dtype: int64

Errant values (i.e., values other than state abbreviations) are in this column.

In [4]:
df_sub = df.copy()

# tidy up strings -- this will remove numeric
df_sub['State'] = df_sub['State'].str.strip()
df_sub['State'] = df_sub['State'].str.upper()

df_sub = df_sub.replace("OHIO", "OH") # abbreviation intended

df_sub['State'] = df_sub['State'].str.slice(0,2) # limit to state abbreviation

df_sub.groupby(['State']).size()

State
CA    880
NY      1
OH      1
dtype: int64

### School

Two obvious issues

    - There are three entries for school per record
    - Typos and superfluous (i.e., non-schoool) information is in each field

In [5]:
# Create subsets of schools, combine

df1 = df_sub['School 1']
df2 = df_sub['School 2']
df3 = df_sub['School 3']

schools = pd.DataFrame(pd.concat([df1, df2, df3]))
schools.rename(columns={0:"school"}, inplace = True)
#schools.value_counts()

schools.value_counts()

school                                        
Citizens of the World - Mar Vista                 80
Palisades Charter High School                     52
Palisades Charter High                            51
Paul Revere                                       42
Paul Revere Charter Middle School                 35
                                                  ..
Kenter Canyon (friend)                             1
Kenter Canyon (starting Fall 2017)                 1
Kenter Canyon Charter School                       1
Kenter Friend--Westwood Charter in a few years     1
plays vista elementary                             1
Length: 264, dtype: int64

In [6]:
# extract everything up to certain punctuation
schools['school'] = schools['school'].str.extract(r'(^[^\(-;,]+)')

# coerce to lowercase, trim whitespace
schools['school'] = schools['school'].str.lower()
schools['school'] = schools['school'].str.strip()

# remove common words
rem_list = [
    'school', 'elementary', 'middle', 'high', 'charter', 'district',
    'employee'
]

schools['school'] = schools['school'].str.replace('|'.join(rem_list), '', regex = False)

schools.value_counts()[1:10]

school                           
warner ave                           66
palisades charter high school        56
palisades charter high               52
kenter canyon                        50
paul revere                          48
paul revere charter middle school    35
wish                                 24
palisades charter elementary         20
open magnet charter school           19
dtype: int64

Further cleaning may require a list of known schools, or some manual combing. Here are the 10 most commonly identified schools.

#### 3. Determine how many missing values you have in each field

In [7]:
df_sub.isnull().sum()

Region                          899
Neighborhood Leadership Team    899
Date Submitted:                 895
First                             0
Last                              0
Email Address                     1
Phone                           398
Address                           4
City                              2
State                            60
Zip                               8
School 1                        224
School 2                        842
School 3                        913
Status                          523
Membership Level                569
Events                          897
Notes                           753
Unnamed: 18                     927
Unnamed: 19                     929
dtype: int64

#### 4. How many bad emails are there? How do you define bad?

A "bad" email, at its most basic level, doesn't match the form `name@company.extension`. In general, as long as both `@` and `.` are present in the string, it represents a good email address. There are certainly exceptions to this rule. 

In [8]:
df_email = pd.DataFrame(df_sub['Email Address'])

df_email['contains_at'] = df_email['Email Address'].str.contains('@')
df_email['contains_dot'] = df_email['Email Address'].str.contains('.')


df_email['email_assess'] = np.where(df_email['contains_at'] & df_email['contains_dot'], 'good', 'bad')

df_email['email_assess'].value_counts()

good    940
bad       2
Name: email_assess, dtype: int64